# CSC790 Notebook 4: Train and publish the demo model

**What this notebook does:** trains **one** AfriBERTa-large sentiment model (seed 42) with exactly the same recipe as Notebook 02, checks it on the full AfriSenti Yorùbá test set, and publishes it to the Hugging Face Hub as
**`Olasquare/afriberta-large-yoruba-sentiment`**, so the demo app (folder `app/`) can use it.

It does **not** change any result in the report. The report gives the mean of 5 seeds; this is a single run, so its score will differ slightly.

**How to run it on Kaggle:**
1. Right-hand panel → **Session options** → Accelerator: **GPU T4** (T4 x2 also works; only one GPU is used) → Internet: **On**.
2. Top menu → **Add-ons → Secrets** → add a secret named **`HF_TOKEN`** holding a Hugging Face token with **write** access, and tick the box so it is attached to this notebook.
3. Top right → **Save Version** → **Save & Run All (Commit)** → **Save**. It takes about 10 to 15 minutes.
4. When the version shows **Successful**, open it: the test scores are printed under **section 6**, and also saved as `test_scores.json` in the Output tab and in the model repo.

## 0. Setup

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn huggingface_hub

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use exactly one GPU (same batch size as the report runs); must be set before torch is imported
import re, csv, json, time
import numpy as np, pandas as pd, torch
import transformers
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, TrainerCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score, classification_report

OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
      "| transformers", transformers.__version__, "| torch", torch.__version__)
if DEVICE == "cpu":
    print("WARNING: no GPU. Turn on GPU T4 in Session options, then run again.")

## 1. Settings

These are the AfriBERTa-large settings from Notebook 02, unchanged.

In [ ]:
HF_REPO = "Olasquare/afriberta-large-yoruba-sentiment"   # where the model is published
BASE_MODEL = "castorini/afriberta_large"
SEED = 42
LR = 5e-5            # learning rate
BATCH = 32           # training batch size (one GPU, no gradient accumulation)
MAX_LEN = 128        # longest tweet length in tokens
MAX_EPOCHS = 10      # upper limit; early stopping usually ends sooner
PATIENCE = 2         # stop after 2 epochs without improvement on the cleaned dev set
WEIGHT_DECAY = 0.01

LABELS = ["negative", "neutral", "positive"]
l2i = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

## 2. Load the data and fix the dev set

Same loading code as Notebooks 01 and 02. The dev set has 270 tweets that also appear in the training set; we drop them (after lower-casing, removing `@user`, links and "rt", and collapsing spaces, exactly as Notebook 02 does) so the dev score used to pick the best epoch is fair.

In [ ]:
BASE = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data/yor"

def load(split):
    d = pd.read_csv(f"{BASE}/{split}.tsv", sep="\t", quoting=csv.QUOTE_NONE, on_bad_lines="skip")
    d["split"] = split
    return d

df = pd.concat([load(s) for s in ["train", "dev", "test"]], ignore_index=True)
df["tweet"] = df["tweet"].astype(str).str.replace('"', '', regex=False).str.strip()

def norm(t):
    t = t.lower()
    t = re.sub(r"@user|https?\S+|\brt\b", "", t)
    return re.sub(r"\s+", " ", t).strip()

df["norm"] = df.tweet.map(norm)
train_norm = set(df.loc[df.split == "train", "norm"])
df["dup_of_train"] = (df.split != "train") & df.norm.isin(train_norm)

train = df[df.split == "train"].reset_index(drop=True)
dev   = df[(df.split == "dev") & ~df.dup_of_train].reset_index(drop=True)
test  = df[df.split == "test"].reset_index(drop=True)
print(f"train {len(train)} | dev {len(dev)} (removed {int(((df.split=='dev') & df.dup_of_train).sum())} duplicates) | test {len(test)}")

## 3. Tokenise and load the base model

The model is loaded in full precision (float32). Mixed-precision training (fp16) still runs on the GPU; loading the weights themselves in fp16 would crash with "Attempting to unscale FP16 gradients".
The label names are stored in the model's config, so the published model answers with "negative", "neutral" or "positive".

In [ ]:
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.model_max_length = 512

def to_ds(d):
    x = d[["tweet"]].copy(); x["labels"] = d.label.map(l2i).values
    return Dataset.from_pandas(x.reset_index(drop=True)).map(
        lambda b: tok(b["tweet"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["tweet"])

tr, dv, te = to_ds(train), to_ds(dev), to_ds(test)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=id2label, label2id=l2i, torch_dtype=torch.float32)
print(model.config.id2label)

## 4. Train

Same as Notebook 02: up to 10 epochs, warm-up over the first 10% of the 10-epoch budget (given as a number of steps), weight decay 0.01, fp16 on the GPU.
After every epoch the model is scored on the cleaned dev set (weighted F1). The best epoch's weights are kept in memory, and training stops after 2 epochs without improvement.

In [ ]:
class KeepBestInMemory(TrainerCallback):
    # Early stopping (patience 2) and keeping the best epoch's weights in memory (copied from Notebook 02).
    def __init__(self, patience=2):
        self.patience, self.best, self.bad, self.state_dict = patience, -1.0, 0, None
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        f = metrics["eval_weighted_f1"]
        if f > self.best:
            self.best, self.bad = f, 0
            self.state_dict = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            self.bad += 1
            if self.bad >= self.patience:
                control.should_training_stop = True

steps_per_epoch = -(-len(train) // BATCH)
warmup = int(0.1 * MAX_EPOCHS * steps_per_epoch)   # 10% of the 10-epoch budget
args = TrainingArguments(
    output_dir="/tmp/ckpt_demo", learning_rate=LR, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=1, per_device_eval_batch_size=64,
    num_train_epochs=MAX_EPOCHS, warmup_steps=warmup, weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch", save_strategy="no", logging_strategy="epoch",
    fp16=(DEVICE == "cuda"), report_to="none", seed=SEED)

def metrics(p):
    return {"weighted_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="weighted")}

keeper = KeepBestInMemory(patience=PATIENCE)
trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=dv, processing_class=tok,
                  compute_metrics=metrics, callbacks=[keeper])
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60

trainer.model.load_state_dict(keeper.state_dict)   # go back to the best epoch
evals = [h for h in trainer.state.log_history if "eval_weighted_f1" in h]
best = max(evals, key=lambda h: h["eval_weighted_f1"])
best_epoch = int(round(best["epoch"]))
print(f"Best epoch: {best_epoch} | cleaned-dev weighted F1: {best['eval_weighted_f1']:.4f} | {train_minutes:.1f} min")

## 5. Evaluate on the full test set

In [ ]:
logits = trainer.predict(te).predictions
pred = [LABELS[i] for i in np.asarray(logits).argmax(-1)]
gold = test.label.tolist()

scores = {
    "weighted_f1": f1_score(gold, pred, average="weighted"),
    "macro_f1": f1_score(gold, pred, average="macro"),
    "accuracy": accuracy_score(gold, pred),
}
report = classification_report(gold, pred, labels=LABELS, digits=3, output_dict=True)

## 6. Test scores of this run

**These are the real test scores of the published model.** They are also saved to `test_scores.json`.

In [ ]:
print(f"Test tweets: {len(test)}")
print(f"Weighted F1: {100*scores['weighted_f1']:.1f}")
print(f"Macro F1:    {100*scores['macro_f1']:.1f}")
print(f"Accuracy:    {100*scores['accuracy']:.1f}\n")
print(classification_report(gold, pred, labels=LABELS, digits=3))

results = {
    "model": HF_REPO, "base_model": BASE_MODEL, "seed": SEED,
    "test_size": len(test), "test": scores, "classification_report": report,
    "best_epoch": best_epoch, "dev_weighted_f1": best["eval_weighted_f1"],
    "dev_size": len(dev), "train_size": len(train), "train_minutes": train_minutes,
    "recipe": {"lr": LR, "batch": BATCH, "max_len": MAX_LEN, "max_epochs": MAX_EPOCHS,
               "warmup_steps": warmup, "weight_decay": WEIGHT_DECAY, "patience": PATIENCE, "fp16": DEVICE == "cuda"},
    "transformers": transformers.__version__, "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
}
with open(f"{OUT}/test_scores.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved", f"{OUT}/test_scores.json")

## 7. Quick check: the model on a few example tweets

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tok, top_k=None,
               device=0 if DEVICE == "cuda" else -1)
for t in ["yay he did it jẹ́ ń lọ gbàlù kí n jó", "ó yá follow me back ẹ ṣeun", "Ọlọ́run á bá wa ṣe é"]:
    r = clf(t, truncation=True, max_length=MAX_LEN)[0]
    print(t, "->", {x["label"]: round(x["score"], 3) for x in r})

## 8. Write the model card

The card shows the scores of **this** run, filled in automatically.

In [ ]:
pc = report  # per-class scores
card = f"""---
language:
- yo
- en
tags:
- sentiment-analysis
- code-mixing
- yoruba
- afrisenti
datasets:
- shmuhammad/AfriSenti-twitter-sentiment
base_model: {BASE_MODEL}
pipeline_tag: text-classification
metrics:
- f1
- accuracy
model-index:
- name: afriberta-large-yoruba-sentiment
  results:
  - task:
      type: text-classification
      name: Sentiment analysis
    dataset:
      name: AfriSenti-SemEval 2023, Yoruba test set
      type: shmuhammad/AfriSenti-twitter-sentiment
      split: test
    metrics:
    - type: f1
      name: Weighted F1
      value: {100*scores['weighted_f1']:.1f}
    - type: f1
      name: Macro F1
      value: {100*scores['macro_f1']:.1f}
    - type: accuracy
      name: Accuracy
      value: {100*scores['accuracy']:.1f}
---

# AfriBERTa-large, Yorùbá tweet sentiment

A 3-class sentiment model (**negative, neutral, positive**) for Yorùbá tweets, including tweets that mix Yorùbá and English.
It is [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}) fine-tuned on the Yorùbá part of AfriSenti-SemEval 2023.
It was trained for the demo of the CSC790 Deep Learning project *Deep Neural Networks for Semantic Sentiment Analysis in Code-Mixed English-Yorùbá Social Media Conversations*.

```python
from transformers import pipeline
clf = pipeline("text-classification", model="{HF_REPO}", top_k=None)
clf("ó yá follow me back ẹ ṣeun", truncation=True, max_length=128)
```

## Training data

[AfriSenti-SemEval 2023](https://github.com/afrisenti-semeval/afrisent-semeval-2023), Yorùbá: {len(train)} training tweets; dev set with tweets that also appear in training removed ({len(dev)} tweets), used only to choose the best epoch; {len(test)} test tweets.
The data is released under **CC BY 4.0** by Muhammad et al. (2023) and builds on NaijaSenti (Muhammad et al., 2022). Please cite them.

## Training recipe

Same as the AfriBERTa-large runs in the project: seed {SEED}, learning rate {LR}, batch size {BATCH}, maximum length {MAX_LEN} tokens, up to {MAX_EPOCHS} epochs, weight decay {WEIGHT_DECAY},
linear warm-up over 10% of the 10-epoch budget ({warmup} steps), fp16 mixed precision, early stopping with patience {PATIENCE} on cleaned-dev weighted F1, keeping the best epoch (epoch {best_epoch} here).
One {results['gpu']} GPU, transformers {transformers.__version__}.

## Test results (this model, full test set of {len(test)} tweets)

| Metric | Score |
|---|---|
| Weighted F1 | {100*scores['weighted_f1']:.1f} |
| Macro F1 | {100*scores['macro_f1']:.1f} |
| Accuracy | {100*scores['accuracy']:.1f} |

| Class | Precision | Recall | F1 | Tweets |
|---|---|---|---|---|
""" + "".join(
    f"| {l} | {100*pc[l]['precision']:.1f} | {100*pc[l]['recall']:.1f} | {100*pc[l]['f1-score']:.1f} | {int(pc[l]['support'])} |\n"
    for l in LABELS) + f"""
The project report gives the **mean of 5 seeds** (AfriBERTa-large: 72.5 weighted F1, AfroXLMR-large: 73.6; the two are statistically tied).
This model is a single run (seed {SEED}), so its score differs slightly from the report.

## Limitations

- Trained and tested on tweets from one platform (Twitter/X) only; other kinds of text may get worse results.
- Its main errors are on **proverbs and idioms**, **sarcasm**, and **slang**.
- Only 3 classes; mixed feelings in one tweet are forced into one label.
- Not for high-stakes decisions about people (for example moderation, hiring or credit). Use it for research and demonstration.

## Authors

Saheed Olayemi Olayinka and Samson Akinyele Fagbenjo, Department of Computer Science, University of Ibadan.
Code: https://github.com/Olasquare043/yoruba-codemix-sentiment
"""
with open(f"{OUT}/README.md", "w", encoding="utf-8") as f:
    f.write(card)
print(card)

## 9. Publish to the Hugging Face Hub

The token is read from **Kaggle Secrets** (never typed into the notebook). The model, tokenizer, model card and `test_scores.json` are uploaded to the repo.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Add a Kaggle secret named HF_TOKEN (Add-ons > Secrets) with a WRITE token, "
                       "attach it to this notebook, and run again.") from e

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)

trainer.model.config.id2label, trainer.model.config.label2id = id2label, l2i
trainer.model.push_to_hub(HF_REPO, token=HF_TOKEN, commit_message="Upload fine-tuned model")
tok.push_to_hub(HF_REPO, token=HF_TOKEN, commit_message="Upload tokenizer")
api.upload_file(path_or_fileobj=f"{OUT}/README.md", path_in_repo="README.md", repo_id=HF_REPO,
                commit_message="Add model card")
api.upload_file(path_or_fileobj=f"{OUT}/test_scores.json", path_in_repo="test_scores.json", repo_id=HF_REPO,
                commit_message="Add test scores")
print("Published: https://huggingface.co/" + HF_REPO)

## 10. Check the published model loads

Downloads the model back from the Hub and classifies one tweet, the same way the demo app will.

In [ ]:
check = pipeline("text-classification", model=HF_REPO, top_k=None, device=-1, token=HF_TOKEN)
print(check("yay he did it jẹ́ ń lọ gbàlù kí n jó", truncation=True, max_length=MAX_LEN))